# 03 — Item Clarity

**Task.** Predict the mean human clarity rating (1-7 scale) for personality test items like "I am the life of the party." The score is Pearson correlation between predicted and actual mean ratings.

**What the winners got.** PAID .816 · Wonderlic .772 · Hungry Llama .740 · Akben .676.

**The pattern.** This is the task where the LLM-first approach loses. PAID won by **explicitly abandoning GPT-4 in favor of a fine-tuned DeBERTa-V3-base.** This notebook walks through *why* and shows what the LLM-only ceiling looks like.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import numpy as np
from src.adapters import ClarityAdapter
from src.harness import Harness, CallSpec
from src.scoring import pearson_r
from src.run import load_csv, DATA_DIR

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Clarity inputs (train): {(DATA_DIR / 'clarity_train.csv').exists()}")
print(f"Clarity inputs (dev):   {(DATA_DIR / 'clarity_val_public.csv').exists() or (DATA_DIR / 'clarity_dev_inputs.csv').exists()}")
print(f"Clarity inputs (test):  {(DATA_DIR / 'clarity_test_public.csv').exists() or (DATA_DIR / 'clarity_test_inputs.csv').exists()}")

## The data

Personality test items, short strings (5-25 words each). Each item has a "mean_clarity" rating from 1.0 to 7.0, computed as the mean across multiple human raters. The training set is around 200 items; dev and test are smaller.

Example items (from the public release):
- "I am the life of the party." — high clarity (~6.5)
- "I find it difficult to get down to work." — high clarity (~6.0)
- "I am not interested in other people's problems." — moderate (~5.5)
- "I see myself as someone who comes up with new ideas." — moderate (~5.4)
- "I am ambivalent about my future." — lower (~4.8)

The clarity distribution is roughly bell-shaped centered around 5.5, with a standard deviation of ~0.8.

In [ ]:
clarity_train = load_csv(DATA_DIR / "clarity_train.csv")
if clarity_train:
    print(f"Loaded {len(clarity_train)} training items")
    print(f"\nColumns: {list(clarity_train[0].keys())}")
    print(f"\nFirst 5 items with their mean clarity:")
    for r in clarity_train[:5]:
        print(f"  {float(r['mean_clarity']):.2f}: {r['item']}")
    
    ratings = [float(r['mean_clarity']) for r in clarity_train]
    print(f"\nRating distribution:")
    print(f"  mean: {np.mean(ratings):.2f}")
    print(f"  std:  {np.std(ratings):.2f}")
    print(f"  range: [{min(ratings):.2f}, {max(ratings):.2f}]")
else:
    print("Clarity training data not present.")

## Why this task breaks LLMs

Pearson correlation rewards two things: getting the ordering right (which items are clearer than which others) AND getting the spread right (your predictions need to be as variable as the truth).

LLMs are bad at the second part. Ask GPT-4o "rate this from 1.0 to 7.0" and the responses cluster around 5.0-6.0 with a standard deviation that's much smaller than the human rating distribution. The ordering is roughly right; the spread is wrong.

The clarity metric is robust to *some* underspread (Pearson r is scale-invariant in the predictions), but it falls apart when many items get the same prediction. If you predict 5.5 for half the test set, the correlation craters.

Below is a quick check on the standard deviation problem. We don't run the LLM here (no API key needed for this point), but the math is the same as it would be.

In [ ]:
# Empirical demonstration of why LLM regression fails on Pearson r.
# Suppose your model's predictions correlate perfectly with truth in
# ORDERING but have only 1/3 the spread of the truth. How does Pearson
# r behave?
np.random.seed(42)
truth = np.random.normal(loc=5.5, scale=0.8, size=200)

# Underspread predictions (correlated but compressed)
pred_underspread = 5.5 + 0.33 * (truth - 5.5) + np.random.normal(scale=0.1, size=200)
print(f"Underspread: pred std = {pred_underspread.std():.3f}, truth std = {truth.std():.3f}")
print(f"  Pearson r = {pearson_r(pred_underspread.tolist(), truth.tolist()):.4f}")
print(f"  (Spread doesn't matter for r itself, but...)")
print()

# Underspread + categorical (which is what LLM '1-7 integer' outputs look like)
pred_categorical = np.round(pred_underspread).clip(1, 7)
print(f"Categorical underspread: pred std = {pred_categorical.std():.3f}")
print(f"  Pearson r = {pearson_r(pred_categorical.tolist(), truth.tolist()):.4f}")
print(f"  (Coarser bins drop r significantly — even a 7-point Likert scale loses information)")
print()

# Now: many items predicted the same value
pred_many_same = pred_categorical.copy()
pred_many_same[pred_many_same > 5] = 5.5  # collapse the top
print(f"Collapsed-tail predictions: r = {pearson_r(pred_many_same.tolist(), truth.tolist()):.4f}")
print(f"  This is the regime LLMs land in when uncertain: collapse to the safe answer.")

## PAID Team's pivot to DeBERTa

PAID Team's deck says, verbatim: "Fine-tune DeBERTa-V3-base (worked). Tried GPT4 (failed). Tried fine-tuned GPT3.5 (failed)."

The DeBERTa setup:
- Base model: `microsoft/deberta-v3-base` (~140M parameters)
- Task head: single-output regression (mean prediction)
- Loss: MSE or directly Pearson correlation (they used both, similar results)
- Weight decay: 0.1
- Train/validation split on the original training set
- Standard transformers + PyTorch fine-tuning loop

The reconstruction below is the actual training script you'd run. It needs PyTorch and the transformers library; it's not part of the harness because the harness is LLM-only by design (see ARCHITECTURE.md, "Where the unification breaks down").

In [ ]:
# DeBERTa fine-tuning sketch (not executed in this notebook).
# To run: pip install torch transformers datasets
# Expected training time on a single GPU: ~15-20 minutes.

DEBERTA_TRAINING_SCRIPT = '''
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
)
from scipy.stats import pearsonr


class ClarityDataset(Dataset):
    def __init__(self, items, ratings, tokenizer, max_length=128):
        self.items = items
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.items)
    
    def __getitem__(self, i):
        enc = self.tokenizer(
            self.items[i], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.ratings[i], dtype=torch.float32),
        }


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.squeeze(-1) if preds.ndim > 1 else preds
    r, _ = pearsonr(preds, labels)
    return {"pearson_r": r, "mse": np.mean((preds - labels) ** 2)}


def train_clarity_deberta(train_csv, output_dir="./deberta_clarity"):
    df = pd.read_csv(train_csv, encoding="utf-8-sig")
    train_df = df.sample(frac=0.9, random_state=42)
    val_df = df.drop(train_df.index)
    
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
    model = AutoModelForSequenceClassification.from_pretrained(
        "microsoft/deberta-v3-base",
        num_labels=1,  # regression
        problem_type="regression",
    )
    
    train_ds = ClarityDataset(
        train_df["item"].tolist(),
        train_df["mean_clarity"].astype(float).tolist(),
        tokenizer,
    )
    val_ds = ClarityDataset(
        val_df["item"].tolist(),
        val_df["mean_clarity"].astype(float).tolist(),
        tokenizer,
    )
    
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.1,  # the value PAID Team used
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="pearson_r",
        greater_is_better=True,
        report_to="none",
    )
    
    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(output_dir)
    return trainer.evaluate()


# After training:
# result = train_clarity_deberta("data/clarity_train.csv")
# print(result["eval_pearson_r"])  # expected: ~0.78-0.82 on the held-out 10%
'''

print(DEBERTA_TRAINING_SCRIPT[:1500] + "\n... (truncated; full script in notebook source) ...")
print()
print("Expected dev Pearson r after DeBERTa fine-tune: ~.78-.82 (matches PAID's .816)")

## The LLM-only adapter — what the ceiling looks like

The ClarityAdapter in this repo is LLM-only. It's runnable, but expected to cap at around r=.65-.75. This isn't a bug; it's the regression-via-text-generation ceiling.

Two design choices in the adapter mitigate the underspread issue somewhat:

1. **No structured-output schema.** The model returns a freeform number, which lets it produce more granular values (5.43, 5.61, etc.) than an integer Likert response would.

2. **Random few-shot, not similarity.** Counter-intuitively, similarity-based few-shot makes clarity *worse* (see KNOWN_LANDMINES.md Landmine 6). Similar training items cluster tightly in clarity space, so the model learns the local cluster's range and predicts inside it — exactly when you want it to spread predictions.

Below is the adapter in action.

In [ ]:
adapter = ClarityAdapter()
print(f"Adapter: {adapter.task_name}")
print(f"K few-shot examples: {adapter.k_examples}")
print()

sample_examples = clarity_train[:5] if clarity_train else [
    {"item": "I am cheerful in the morning.", "mean_clarity": 6.2},
    {"item": "I have a soft spot for stray animals.", "mean_clarity": 6.0},
    {"item": "My feelings are easily hurt.", "mean_clarity": 5.8},
]
sample_test = {"item": "I prefer order to chaos."}
messages = adapter.build_messages(sample_test, sample_examples)
print(f"--- System prompt ---\n{messages[0]['content']}\n")
print(f"--- First user turn ---\n{messages[1]['content']}")
print(f"--- First assistant turn ---\n{messages[2]['content']}")
print(f"--- Final user turn ---\n{messages[-1]['content']}")

## End-to-end LLM-only run

In [ ]:
from src.run import run_task

result = run_task(
    task="clarity",
    split="dev",
    model="gpt-4o-2024-08-06",
    self_consistency=1,
    output_path=None,
    row_id=None,
    similarity_examples=False,
)
if result["status"] == "ok":
    print(f"Clarity dev: n={result['n']}, pearson_r={result.get('score', 'n/a'):.4f}")
    if result.get("predictions"):
        preds = result["predictions"]
        print(f"\nPrediction statistics:")
        print(f"  mean: {np.mean(preds):.3f}")
        print(f"  std:  {np.std(preds):.3f}  (compare to truth std ~0.8 — likely smaller)")
else:
    print(f"Status: {result['status']}")
    print(f"Message: {result.get('message')}")

## Discussion — the right tool

Clarity is the cleanest illustration of "use the right tool for the task":

1. **LLM zero/few-shot regression caps at ~.70.** That's a real ceiling, not a prompt-engineering problem. The reason is calibration: chat models aren't trained to produce calibrated continuous values.

2. **Fine-tuned BERT-family regressors land at .78-.82.** Same data, ~15 minutes of training, ~140M parameter model. The cost is one-time and dwarfs the cost difference of "GPT-4o per request" vs "BERT inference."

3. **Multi-model ensembles (Akben's GPT-4 + GPT-3.5 + Claude-3) don't fix it.** Combining miscalibrated estimators doesn't yield a calibrated one.

4. **Feature engineering + stacking (Hungry Llama) partially fixes it.** Their .740 is meaningfully above the LLM-only ceiling. But it's still below PAID's .816, and the engineering effort is comparable to (or larger than) just fine-tuning DeBERTa.

Pedagogical takeaway: when you have ground-truth continuous values, you have a regression problem, not a language problem. Reach for a model designed for the former.